# M3 · RAG con tool use — triaje de acciones constitucionales

M2 dejó debilidades medidas, no supuestas, y son exactamente lo que un RAG
puede cerrar: **recuperar el texto de la norma en vez de recordarlo**, y
**recuperar la causal de improcedencia en vez de deducirla**.

Se comparan cuatro sistemas con el mismo harness de M2:

| | sistema | qué aporta |
|---|---|---|
| 1 | fine-tuned solo | baseline de M1, ya medido |
| 2 | fine-tuned + RAG ingenuo | punto de partida del delta de retrieval |
| 3 | fine-tuned + RAG avanzado | la estrategia que gane la regla de decisión |
| 4 | agente con tool use + motor de reglas | el techo: enrutamiento determinista |

**La sección de recuperación corre en CPU.** Solo los brazos 2 y 3 necesitan
GPU, porque cargan el modelo ajustado. Sin GPU el notebook mide recuperación,
herramientas y el brazo 4, y lo dice en vez de fallar.

In [1]:
# RAGAS SE INSTALA AQUI, EN LA PRIMERA CELDA, Y NO DONDE SE USA.
#
# `ragas` arrastra `langchain-core` y `pydantic`. Instalarlo en la celda 34 ---
# donde esta la metrica --- significa pedirle a pip que actualice paquetes que
# `torch`, `datasets` y `sentence_transformers` YA importaron, y entonces los
# modulos cargados siguen siendo los viejos: el fallo sale como un error de
# atributo dentro de ragas, lejos de su causa y despues de 40 minutos de
# brazos. Aqui, si hace falta reiniciar, no se ha perdido nada.
#
# EL PIN QUE IMPORTA ES EL DE `langchain-community`, Y COSTO CUATRO
# CORRIDAS AVERIGUARLO.
#
# Las cuatro registraron el mismo error:
#   No module named 'langchain_community.chat_models.vertexai'
#
# Durante tres se atribuyo a la version de `ragas` y se pineo `ragas==0.2.15`.
# La cuarta salio IDENTICA, asi que esa hipotesis era falsa. La causa, leida
# del traceback y reproducida en un entorno aislado: `ragas/llms/base.py`
# linea 8 hace un import INCONDICIONAL de
# `langchain_community.chat_models.vertexai`, y `langchain-community` 0.4.x
# elimino ese modulo. `0.3.27` es la ultima que lo tiene.
#
# Comprobado antes de volver a gastar una corrida: con 0.3.27 importan
# `ragas.evaluate`, `ragas.metrics.faithfulness`, `LangchainLLMWrapper` y
# `ChatOpenAI` apuntando a Groq, y `Dataset.from_dict` acepta las cuatro
# columnas antiguas. Lo unico que no se pudo probar en local es la llamada.
# `langchain-openai` es el envoltorio que ragas acepta, y hace falta porque el
# juez de ragas se le pasa EXPLICITO: sin `llm=`, `evaluate()` apunta a OpenAI
# por omision y este proyecto solo tiene clave de Groq.
# LOS PINS SON LOS DE `requirements.txt`, Y ESO ERA UN AGUJERO.
#
# `sentence-transformers` y `rank_bm25` iban SIN version. `requirements.txt`
# las fija --- 5.7.0 y 0.2.2, leidas de `corridas.versiones()` de las
# corridas que produjeron las cifras del reporte --- y este notebook se
# traia la ultima. El embedder decide la recuperacion y BM25 decide el
# baseline lexico, asi que una version distinta mueve la tabla de retrieval
# y nada en la salida lo diria: `corrida.json` registra la version que
# HUBO, no la que debia haber.
!pip install -q groq==1.7.0 sentence-transformers==5.7.0 rank_bm25==0.2.2 ragas==0.2.15 "langchain-community==0.3.27" langchain-openai 2>/dev/null

# Colab trae torchao 0.10.0 y peft >=0.21 exige >0.16: al cargar el adaptador
# LoRA lanza ImportError, y lo hace DESPUES de haber bajado y cargado los 15 GB
# del modelo base. Aqui no se usa torchao (solo hace falta para cuantizacion y
# el brazo 3 corre en bf16), asi que se desinstala antes de que nada lo importe.
# El mismo arreglo esta en la celda de instalacion de M1.
!pip uninstall -y -q torchao

import importlib.util
assert importlib.util.find_spec("torchao") is None, (
    "torchao sigue instalado: Entorno -> Reiniciar sesion y corre esta celda de nuevo")

# Que ragas quedo instalado Y EN LA VERSION PINEADA se comprueba ya, no en la
# celda 34. Un aviso aqui cuesta un reinicio; alli cuesta la corrida.
RAGAS_PIN = "0.2.15"
try:
    import ragas as _ragas
    _v = _ragas.__version__
    print(f"ragas {_v}" + ("" if _v.startswith("0.2.") else
                           f"  <- NO es {RAGAS_PIN}: el contraste de la "
                           "seccion 7 fallara"))
except Exception as _ex:
    print(f"ragas NO quedo instalado ({type(_ex).__name__}: {str(_ex)[:90]}).")
    print("   Entorno -> Reiniciar sesion y corre esta celda otra vez. Si")
    print("   insiste, la seccion 7 publica las metricas propias y declara")
    print("   que el contraste con la libreria no se pudo correr.")

import os, sys, pathlib, time, json
import torch

HAY_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if HAY_GPU else "no hay — se omiten los brazos 2 y 3")

ragas NO quedo instalado (ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai').
   Entorno -> Reiniciar sesion y corre esta celda otra vez. Si
   insiste, la seccion 7 publica las metricas propias y declara
   que el contraste con la libreria no se pudo correr.
GPU: NVIDIA A100-SXM4-40GB


## 0 · Proyecto, clave y corrida

In [2]:
CARPETA = "agenteAmparo"

MONTAJE = pathlib.Path("/content/drive")

# El montaje se reintenta y, si no lo consigue, SE PARA AQUI. Antes la
# excepcion se atrapaba y la celda seguia, asi que un fallo de credenciales
# acababa apareciendo como "No encuentro el proyecto": el sintoma, no la
# causa, y manda a revisar rutas cuando lo que fallo fue la autorizacion.
try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    for intento in (1, 2):
        try:
            drive.mount(str(MONTAJE), force_remount=(intento == 2))
        except Exception as ex:
            print(f"intento {intento} de montar Drive fallo: {ex}")
            time.sleep(3)
        # No basta con que mount() no lance: hay que ver el punto de montaje.
        if (MONTAJE / "MyDrive").is_dir():
            print(f"Drive montado (intento {intento})")
            break

    if not (MONTAJE / "MyDrive").is_dir():
        raise SystemExit(
            "Drive NO quedo montado. Es un fallo de autorizacion, no del proyecto.\n"
            "  1. Vuelve a correr ESTA celda y acepta la ventana de permisos.\n"
            "     Si no aparece, mira si el navegador la esta bloqueando.\n"
            "  2. Si insiste: Entorno -> Reiniciar sesion y corre la celda de nuevo.\n"
            "  3. Comprueba que estas en la cuenta de Google dueña de agenteAmparo.")

CANDIDATAS = [
    pathlib.Path("/content/drive/MyDrive") / CARPETA,
    pathlib.Path("/content/drive/MyDrive/Colab Notebooks") / CARPETA,
    pathlib.Path("."),
    pathlib.Path("/content") / CARPETA,
    pathlib.Path(".."),
]
RAIZ = next((c.resolve() for c in CANDIDATAS if (c / "src" / "schema.py").exists()), None)
if RAIZ is None:
    # Con Drive ya montado, el problema es la carpeta: se lista lo que hay de
    # verdad en MyDrive en vez de dejar al lector adivinando el nombre.
    print("Drive montado:", (MONTAJE / "MyDrive").is_dir())
    print("Busque src/schema.py en:")
    for cand in CANDIDATAS:
        print("   -", cand, "<- la carpeta existe" if cand.is_dir() else "")
    if (MONTAJE / "MyDrive").is_dir():
        print("\nLo que hay en MyDrive:")
        for p in sorted((MONTAJE / "MyDrive").iterdir())[:30]:
            print("   -", p.name)
    raise SystemExit("No encuentro el proyecto")

sys.path.insert(0, str(RAIZ / "src"))
os.chdir(RAIZ)

try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("clave de Groq cargada")
except Exception as ex:
    print("sin secreto de Colab:", ex)
    import getpass
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")

import schema, metricas, harness, rag, reglas, terminos, sistema, ragas_eval, agente, corridas

c = corridas.nueva("m3")
print("raiz:", RAIZ)
print("datos:", c.meta["datos"])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive montado (intento 1)
clave de Groq cargada
corrida m3/20261021T140413Z · runs/m3/20261021T140413Z
raiz: /content/drive/MyDrive/Colab Notebooks/agenteAmparo
datos: {'dataset.json': {'version': '3.5.0', 'n': 160, 'bytes': 249885}, 'eval_set.json': {'version': '1.2.0', 'n': 46, 'bytes': 80388}, 'eval_set_v2.json': {'version': '2.0.0', 'n': 69, 'bytes': 113139}, 'eval_set_v3.json': {'version': '3.0.0', 'n': 109, 'bytes': 175078}, 'eval_set_m3.json': {'version': '1.1.0', 'n': 21, 'bytes': 39778}, 'corpus.json': {'version': '1.5.1', 'n': 69, 'bytes': 125864}}


In [3]:
# ¿Drive tiene lo MISMO que el disco local?
#
# `arranque.celda()` comprueba que los archivos EXISTAN. Esto comprueba que
# sean los MISMOS, que es otra cosa: un `metricas.py` viejo existe igual de
# bien que el nuevo, y una corrida sobre `src/` desactualizado produce cifras
# perfectamente plausibles y perfectamente inservibles sin que nada lo delate.
#
# Va aqui y no mas abajo porque la celda del modelo baja 15 GB de pesos:
# enterarse antes cuesta dos segundos, enterarse despues cuesta la corrida.
#
# COMPARA CONTRA EL SELLO EN VEZ DE IMPRIMIR Y CONFIAR.
#
# Antes imprimia la huella y le pedia a quien mira que la comparase con la del
# disco local. Tres fallos de una sola sesion vinieron de ahi: una celda que no
# se subio, una que se subio a medias, y una huella que no cuadraba y se
# arranco igual. `data/subida_sellada.txt` --- que lo genera
# `preparar_subida.py --sellar` y se sube con todo lo demas --- dice que
# DEBERIA haber, asi que la comparacion la hace el codigo y nombra el archivo.
#
# Con `subprocess` y no con `!python`, por dos razones: la ruta absoluta de
# Drive lleva un espacio en "Colab Notebooks" y el shell la partiria, y una
# celda con magia no la puede comprobar la autoprueba --- la prueba 41 parsea
# las celdas de codigo y una magia precedida de comentarios la rompe.
import subprocess
import sys as _sys

_m = subprocess.run([_sys.executable, "-I", "data/manifiesto.py",
                     "--contra-sello"], capture_output=True, text=True)
print(_m.stdout.strip() or (_m.stderr or "")[-400:])
if _m.returncode == 1:
    raise SystemExit(
        "Hay archivos sin subir o desincronizados (arriba estan nombrados). "
        "Subelos y vuelve a correr esta celda ANTES de seguir.")
if _m.returncode == 2:
    print()
    print("AVISO: sin sello no se puede verificar que Drive este al dia.")
    print("Compara la huella de arriba con la del disco local a mano.")

_TOTAL  2332889 bytes  dbe1c353a156c359
COINCIDE con subida_sellada.txt: 71 archivos, todos al dia.


In [4]:
# ROADMAP #31 y #33. Son 45 minutos: vale la pena gastar dos segundos en saber
# que el entorno esta listo. Comprueba el corpus, las predicciones de M1 y la
# clave de Groq, y recuerda donde va este modulo en el orden de los tres.
import arranque
arranque.celda("m3")

  ARRANQUE · M3   (TERCERO de los tres)
  entorno   : GPU (A100 recomendada)
  duracion  : ~45 min
  necesita  : results/m1_predicciones.json, GROQ_API_KEY, data/corpus.json
  produce   : 13 artefactos en results/

  No depende de M2. Guarda checkpoint entre brazos, asi que un fallo en el brazo 4 no se lleva los anteriores.

  orden de los tres modulos:
    PRIMERO  M1   GPU (A100 recomendada)    ~30 min: ~10 de entrenamiento con gradient_checkpointing, ~4 de las dos generaciones por lotes, y ~14 del brazo hibrido, que genera caso a caso (lote 1) para no falsear la latencia que mide `costos`
    SEGUNDO  M2   CPU basta                 ~3 min
    TERCERO  M3   GPU (A100 recomendada)    ~45 min <-- estas aqui

  prueba de humo: 16 comprobaciones pasan
  entorno listo.


{'ok': True,
 'fallos': [],
 'avisos': [],
 'pasa': ['data/dataset.json',
  'data/eval_set.json',
  'data/eval_set_v2.json',
  'data/eval_set_v3.json',
  'data/eval_set_m3.json',
  'data/corpus.json',
  'data/normas.json',
  'import schema',
  'import metricas',
  'import corridas',
  'import reglas',
  'results/m1_predicciones.json',
  'GROQ_API_KEY presente',
  'GPU: NVIDIA A100-SXM4-40GB',
  'torch 2.11.0+cu130 / transformers 5.20.1',
  'torchao ausente (bien: choca con transformers)']}

## 1 · Criterio 1 — Corpus: procedencia y responsabilidad

**De dónde salió.** Transcrito del normograma oficial de la Cancillería y de la
edición de la Rama Judicial de la Constitución, extraído **con script** y sin
que nadie lo retipeara: `data/reparar_corpus.py` y `data/auditar_corpus.py`
quedan en el repo y son re-ejecutables.

**Licencia.** Normas colombianas: documentos oficiales de dominio público,
Ley 23 de 1982 artículo 41.

**Quién responde.** El equipo. La transcripción es nuestra, no una copia
certificada.

**Qué pasa si el corpus está mal.** Un RAG sirve lo que recupera con apariencia
de autoridad. Por eso cada fragmento lleva URL, `vigencia`, `transcripcion` y
`verificado`, y los incisos inexequibles se **marcan en vez de borrarse**: el
modelo tiene que poder recuperar el art. 11 del Decreto 2591 y contradecir a
quien lo invoque como vigente.

In [5]:
import collections

docs = rag.cargar_corpus()
print(f"{len(docs)} fragmentos · {sum(len(d['texto']) for d in docs)} caracteres")
print(f"verificados contra fuente oficial : {sum(d['verificado'] for d in docs)}/{len(docs)}")
print(f"transcripciones literales         : "
      f"{sum(d['transcripcion'] == 'literal' for d in docs)}/{len(docs)}")
print()
for norma, n in collections.Counter(d["norma"] for d in docs).most_common():
    print(f"  {n:>3}  {norma}")
print()
print("--- fragmentos con nota de vigencia que NO es 'vigente' ---")
for d in docs:
    if d["vigencia"] != "vigente":
        print(f"  {d['id']:<12}{d['vigencia']}")

69 fragmentos · 61813 caracteres
verificados contra fuente oficial : 69/69
transcripciones literales         : 68/69

   20  Constitución Política de Colombia
   17  Decreto 2591 de 1991
    5  Ley 1095 de 2006
    5  Ley 472 de 1998
    4  Ley 1751 de 2015
    4  Ley 393 de 1997
    3  Ley 1755 de 2015
    2  Ley 100 de 1993
    2  Ley 1581 de 2012
    1  Sentencia C-531 de 2000
    1  Sentencia C-744 de 2012
    1  Sentencia T-225 de 1993
    1  Ley 1098 de 2006
    1  Ley 1266 de 2008
    1  Ley 1437 de 2011
    1  Ley 361 de 1997

--- fragmentos con nota de vigencia que NO es 'vigente' ---
  D2591-06    vigente con un inciso inexequible
  D2591-11    INEXEQUIBLE (C-543 de 1992)
  L1095-01    vigente; articulo CONDICIONALMENTE exequible (C-187 de 2006)
  L1098-111   vigente; numeral 5 derogado por la Ley 1564 de 2012
  L1266-13    vigente, modificado por el art. 3 de la Ley 2157 de 2021
  L1751-08    vigente; el paragrafo del articulo fue declarado INEXEQUIBLE
  L1755-13    vigente;

### Tabla de riesgos del corpus

| riesgo | ejemplo concreto aquí | qué le pasa al usuario si nadie lo detecta | cómo se detecta | cómo se mitiga |
|---|---|---|---|---|
| norma derogada citada como vigente | `D2591-11`, art. 11, **inexequible** por C-543/1992 | cree que su tutela caducó a los dos meses y no la presenta | `vigencia` distinto de "vigente" | se marca y se deja en el corpus para poder contradecirla; está **fuera** del whitelist de `normas.json` |
| paráfrasis servida como ley | en la v1.0.0 había cinco: `L1098-ALIM`, `L1437-NYR`, `L1581-SENS`… | cita como texto legal algo que escribió el equipo | `transcripcion` distinto de "literal" | las cinco se reemplazaron por transcripción literal; hoy el corpus es 100% literal |
| plazo mal entendido | Ley 1266 art. 13: **coexisten** 4 años desde el pago y 8 desde la mora | pide que le borren el reporte con el plazo equivocado | cotejo contra el texto oficial | `nota_vigencia` explica los dos disparadores |
| modificación no incorporada | art. 26 Ley 361: la copia oficial sigue diciendo "oficina de Trabajo" | invoca una autoridad que ya no es la competente | comparación con la norma modificatoria | marcado como **pendiente de verificar** en vez de afirmarlo |
| restos del conversor en el texto | marcadores "Jurisprudencia Unificación" y frases duplicadas en `L472-04` | lee como ley una etiqueta de navegación | `data/auditar_corpus.py` | auditoría re-ejecutable de los 57 fragmentos |
| cita sin respaldo en el corpus | `Ley 2157 de 2021`, `art. 42 CP`, `Ley 100 de 1993` | el sistema cita una norma que el RAG no puede mostrar | chequeo de cobertura en el auditor | las tres entraron; el chequeo corre en cada auditoría |
| sesgo de cobertura | 19 de 57 fragmentos son de la Constitución y 17 del Decreto 2591 | el sistema razona mejor sobre procedimiento que sobre derechos sustantivos | distribución por norma | declarado; es la debilidad principal del corpus |

**Fuentes descartadas y por qué.** Las copias en blogs y agregadores jurídicos
quedaron fuera porque no permiten verificar la vigencia; `secretariasenado.gov.co`
quedó fuera porque no respondió al momento de la cosecha, y se usó el
normograma de la Cancillería, que ya era la fuente de las leyes 1751, 1755 y 393.

In [6]:
# Auditoria del corpus + chequeo de cobertura, ejecutados aqui para que el
# resultado quede en la salida del notebook y no solo en la consola de alguien.
import subprocess
print(subprocess.run([sys.executable, "data/auditar_corpus.py"],
                     capture_output=True, text=True).stdout)

cobertura: 77 chunks contra las citas de los datasets
  toda cita de los datasets tiene un fragmento que la respalda

auditoria de 69 fragmentos del corpus
  limpio: ningun resto del conversor, ninguna frase duplicada



## 2 · Chunking: por artículo o por tamaño fijo

La hipótesis a batir es que el artículo es la unidad correcta, porque es la
unidad de cita en derecho colombiano: un fragmento que corta un artículo por la
mitad produce una cita que no se puede verificar.

Eso es una convicción hasta que se mide. La regla de decisión está fijada en
`rag.REGLA_CHUNKING`, **antes** de correr.

In [7]:
print(rag.REGLA_CHUNKING)

Regla de decision del chunking, FIJADA ANTES DE CORRER
(version 2026-10-15):

  1. Mayor hit@3.
  2. Empate (<= 0.01): mayor mrr@10.
  3. Empate: menos chunks (un indice mas pequeno es mas barato y mas auditable).

La hipotesis a batir es que el chunking POR ARTICULO gana, porque el articulo es
la unidad de cita en derecho colombiano: un fragmento que corta un articulo por
la mitad produce una cita que no se puede verificar. Se compara contra tamano
fijo para que esa hipotesis sea una medicion y no una conviccion.


In [8]:
# El nombre VIAJA A LOS ARTEFACTOS, asi que no se escribe dos veces: el
# artefacto `ganadores` decia "los N casos de eval_set_v2.json"
# mientras
# esta celda ya cargaba otro archivo. Una constante no puede discrepar.
ARCHIVO_EVAL = "eval_set_v3.json"
eval_set = schema.cargar(archivo=ARCHIVO_EVAL)
eval_m3 = schema.cargar(archivo="eval_set_m3.json")
print(f"eval compartido: {len(eval_set)} casos · eval solo-M3: {len(eval_m3)} casos")

CONFIGS = {
    "articulo (nuestro)": rag.trocear,
    "fijo 400/60": rag.trocear_fijo,
    "fijo 900/120": lambda d: rag.trocear_fijo(d, 900, 120),
}
dec_chunk = rag.comparar_chunking(docs, eval_set, CONFIGS, estrategia="hibrido", k=5)
c.guardar_json("chunking", dec_chunk)

eval compartido: 109 casos · eval solo-M3: 21 casos


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


  CHUNKING · misma estrategia (hibrido), mismo k
config                      chunks   chars    hit@1    hit@3   mrr@10  recall@k  lat p95
--------------------------------------------------------------------------------------
articulo (nuestro)              77     803    26.2%    43.7%    0.369     24.9%     26.6
fijo 400/60                    216     325    27.2%    39.8%    0.357     23.5%     26.5
fijo 900/120                   117     572    27.2%    39.8%    0.362     26.5%     25.6
  Regla de chunking fijada antes de correr:
    mejor hit@3 = 43.7% · 1 config(s) en la tolerancia
    ELEGIDO: articulo (nuestro)


PosixPath('/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/chunking.json')

In [9]:
chunks = CONFIGS[dec_chunk["elegido"]](docs)
indice = rag.Indice(chunks).construir()
print(f"indice: {len(chunks)} chunks con el chunking '{dec_chunk['elegido']}'")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

índice listo · 77 fragmentos · denso intfloat/multilingual-e5-base
indice: 77 chunks con el chunking 'articulo (nuestro)'


## 3 · Criterio 2 — Cinco técnicas de retrieval, con el delta medido

Lo único que cambia entre filas es el retrieval: mismo corpus, mismo chunking,
mismo k, mismas consultas de entrada.

- **Ingenuo**: embeddings densos, coseno, top-k. El baseline a batir.
- **Híbrido**: BM25 + denso fusionados con RRF. Fusiona por *puesto*, no por
  puntaje, así que no hay que normalizar escalas incomparables.
- **Reranking**: un cross-encoder reordena los candidatos leyendo consulta y
  fragmento juntos.
- **Transformación de consulta**: reescribe el relato del ciudadano en lenguaje
  jurídico antes de buscar. Es la que más debería rendir aquí, porque el vacío
  de vocabulario es el cuello de botella.
- **Multi-consulta**: tres reescrituras desde ángulos distintos, fusionadas con
  RRF. Basta con que una acierte el vocabulario de la norma.
- **HyDE**: genera el artículo hipotético que resolvería el caso y busca con
  ese texto.

La regla de decisión también está fijada de antemano, con una **guarda de
seguridad**: una técnica que recupere menos que el ingenuo queda descalificada
por elegante que sea.

In [10]:
print(rag.REGLA_DECISION)

Regla de decision de la estrategia de recuperacion, FIJADA ANTES DE CORRER
(version 2026-10-15, anterior a la primera corrida de M3):

  0. GUARDA DE SEGURIDAD. Una estrategia queda descalificada si su recall@k es
     INFERIOR al del RAG ingenuo. Una tecnica avanzada que recupera menos que el
     baseline no entra al sistema por mas elegante que sea.
  1. Mayor recall@k.
  2. Empate (diferencia <= 0.01): mayor mrr@10 -- premia traer lo relevante arriba.
  3. Empate: menor latencia p95 del retrieval.
  4. Empate: menos casos sin recuperar ningun articulo esperado.

Se escribe aqui, en el codigo y con fecha, para que la eleccion no se haga
despues de ver los numeros. Es la diferencia entre elegir una estrategia y
justificar la que salio mejor.


In [11]:
class CostoPorBrazo:
    """Envuelve un transformador para tres cosas:

    1. DARLE `.costo()`. `evaluar_recuperacion` lo llama siempre, y los brazos
       de multiconsulta y HyDE no reciben el objeto sino un lambda y un metodo
       enlazado, que no lo tienen: AttributeError a mitad de la evaluacion.

    2. REPARTIR EL COSTO. Los tres brazos con transformacion comparten el
       mismo `tr`, cuyos contadores son acumulativos. Sin esto, el brazo de
       multiconsulta reportaria su costo mas el del anterior, y el de HyDE los
       tres juntos: numeros inflados con apariencia de medidos.

    3. CRONOMETRAR AQUI, que uniformiza. Cada tecnica podria registrar su
       latencia de forma distinta --- `multiples()` no la registraba en
       absoluto y el brazo salia con latencia None, pareciendo gratis siendo
       el que hace una llamada EXTRA por caso, y el que gano la comparacion.
       Midiendo en el envoltorio, los tres se cronometran igual.

    Se cronometra SOLO cuando hubo llamada real: mezclar los aciertos de cache
    bajaria la media por una razon falsa.
    """

    def __init__(self, fn, dueno):
        self._fn, self._d = fn, dueno
        self._llamadas0 = dueno.llamadas
        self._fallos0 = dueno.fallos
        self._usos = 0
        self._lat: list[float] = []

    def __call__(self, relato):
        n0 = self._d.llamadas
        t0 = time.perf_counter()
        out = self._fn(relato)
        ms = (time.perf_counter() - t0) * 1000
        self._usos += 1
        if self._d.llamadas > n0:
            self._lat.append(ms)
        return out

    def costo(self) -> dict:
        L = sorted(self._lat)
        reales = self._d.llamadas - self._llamadas0
        return {
            "llamadas_reales": reales,
            "desde_cache": self._usos - reales,
            "fallos": self._d.fallos - self._fallos0,
            "latencia_ms_media": round(sum(L) / len(L), 1) if L else None,
            "latencia_ms_p95": round(L[int(0.95 * (len(L) - 1))], 1) if L else None,
            "transformaciones": self._usos,
        }


tr = rag.TransformadorConsulta(cache=c.ruta_cache("consultas.json"))

resultados = [
    rag.evaluar_recuperacion(indice, eval_set, "ingenuo",  k=5),
    rag.evaluar_recuperacion(indice, eval_set, "hibrido",  k=5),
    rag.evaluar_recuperacion(indice, eval_set, "reranked", k=5),
    rag.evaluar_recuperacion(indice, eval_set, "reranked", k=5,
                             transformador=CostoPorBrazo(tr, tr)),
    rag.evaluar_recuperacion(indice, eval_set, "hibrido",  k=5,
                             transformador=CostoPorBrazo(lambda r: tr.multiples(r, 3), tr)),
    rag.evaluar_recuperacion(indice, eval_set, "hibrido",  k=5,
                             transformador=CostoPorBrazo(tr.hyde, tr)),
]
# El brazo 3 tambien lleva nombre propio: sin el se quedaba como "reranked" y
# chocaba en la tabla con el brazo sin transformacion.
resultados[3]["estrategia"] = "reranked + reescritura"
resultados[4]["estrategia"] = "hibrido + multiconsulta"
resultados[5]["estrategia"] = "hibrido + hyde"
tr.guardar()

print("costo de transformacion por brazo (no acumulado):")
for r in resultados[3:]:
    kk = r["costo_transformacion"]
    print(f"  {r['estrategia']:<26} {kk['llamadas_reales']:>3} reales + "
          f"{kk['desde_cache']:>3} cache de {kk['transformaciones']} · "
          f"{kk['latencia_ms_media']} ms media / {kk['latencia_ms_p95']} ms p95")
print("acumulado del transformador compartido:", tr.costo())
print()

decision = rag.comparar_estrategias(resultados, k=5)
c.guardar_json("retrieval", [{k2: v for k2, v in r.items() if k2 != "detalle"}
                             for r in resultados])
c.guardar_json("decision_retrieval", decision)

config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

configuration_xlm_roberta.py:   0%|          | 0.00/2.73k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-reranker-v2-base-multilingual:
- configuration_xlm_roberta.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


modeling_xlm_roberta.py:   0%|          | 0.00/43.8k [00:00<?, ?B/s]

embedding.py:   0%|          | 0.00/2.56k [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

  reranker: BAAI/bge-reranker-base
costo de transformacion por brazo (no acumulado):
  reranked + reescritura       0 reales + 103 cache de 103 · None ms media / None ms p95
  hibrido + multiconsulta      0 reales + 103 cache de 103 · None ms media / None ms p95
  hibrido + hyde               0 reales + 103 cache de 103 · None ms media / None ms p95
acumulado del transformador compartido: {'llamadas_reales': 0, 'desde_cache': 362, 'fallos': 0, 'latencia_ms_media': None, 'latencia_ms_p95': None}


  RECUPERACION · tabla de deltas
estrategia                      recall@k   prec@k   hit@1   hit@3   mrr@10   lat ms     p95  sin nada    delta
--------------------------------------------------------------------------------------------------------
ingenuo                           30.6%    20.0%   25.2%   44.7%    0.376     23.7    24.4        50         
hibrido                           24.9%    14.2%   26.2%   43.7%    0.369     24.3    24.9        52    -5.7%
reranked                     

PosixPath('/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/decision_retrieval.json')

### Consultas fallidas

Los casos donde el retrieval no trajo ni un artículo esperado. Es la tarea de
S07 y el insumo de S08: cada fallo se le asigna una técnica.

In [12]:
peor = resultados[0]
elegida = decision["elegida"]
mejor = next(r for r in resultados if r["estrategia"] == elegida)

for etiqueta, r in (("INGENUO", peor), (f"ELEGIDA ({elegida})", mejor)):
    fallos = [d for d in r["detalle"] if d["recall"] == 0]
    print(f"{etiqueta}: {len(fallos)}/{r['n']} casos sin recuperar ningun articulo esperado")
    for d in fallos[:6]:
        e = next(x for x in eval_set if x["id"] == d["id"])
        print(f"   {d['id']:<5} {e['categoria']:<30} esperaba {d['esperadas']}")
        print(f"         obtuvo  {d['obtenidas'][:3]}")
    print()

arreglados = {d["id"] for d in peor["detalle"] if d["recall"] == 0} - \
             {d["id"] for d in mejor["detalle"] if d["recall"] == 0}
# Y los que ROMPIO. Contar solo los arreglados es quedarse con la mitad buena:
# una tecnica puede subir el recall global y aun asi perder casos que el
# ingenuo si recuperaba, y eso hay que verlo. Mismo criterio que medir la
# abstencion en los dos sentidos.
rotos = {d["id"] for d in mejor["detalle"] if d["recall"] == 0} - \
        {d["id"] for d in peor["detalle"] if d["recall"] == 0}

print(f"casos que la elegida ARREGLO respecto del ingenuo: {sorted(arreglados)}")
print(f"casos que la elegida ROMPIO respecto del ingenuo : {sorted(rotos)}")

c.guardar_json("consultas_fallidas", {
    "estrategia_elegida": elegida,
    "ingenuo": [d for d in peor["detalle"] if d["recall"] == 0],
    "elegida": [d for d in mejor["detalle"] if d["recall"] == 0],
    "arreglados_por_la_elegida": sorted(arreglados),
    "rotos_por_la_elegida": sorted(rotos)})

INGENUO: 50/103 casos sin recuperar ningun articulo esperado
   E03   peticion_respuesta_evasiva     esperaba ['Ley 1755 de 2015', 'art. 23 CP']
         obtuvo  ['Ley 1266 de 2008', 'Ley 1098 de 2006', 'Ley 100 de 1993']
   E06   debido_proceso_disciplinario   esperaba ['art. 29 CP', 'art. 42 Decreto 2591 de 1991']
         obtuvo  ['Ley 1266 de 2008', 'art. 48 CP', 'art. 30 Decreto 2591 de 1991']
   E07   educacion_inclusion            esperaba ['art. 13 CP', 'art. 44 CP', 'art. 67 CP']
         obtuvo  ['Ley 1751 de 2015', 'Ley 1751 de 2015', 'Ley 1755 de 2015']
   E09   colectivo_ambiente             esperaba ['Ley 472 de 1998', 'art. 6.3 Decreto 2591 de 1991', 'art. 79 CP']
         obtuvo  ['Sentencia C-531 de 2000', 'art. 49 CP', 'Ley 1266 de 2008']
   E10   contractual_ordinario          esperaba ['art. 6.1 Decreto 2591 de 1991']
         obtuvo  ['Ley 1266 de 2008', 'Ley 1098 de 2006', 'Ley 1437 de 2011']
   E19   habeas_data_publicacion        esperaba ['Ley 1581 de 2012', 'a

PosixPath('/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/consultas_fallidas.json')

## 4 · Criterio 3 — Tool use

Dos herramientas, y el **modelo decide cuál llamar**: no las dispara un `if`.

- `calcular_termino` — la herramienta externa. Un LLM no sabe que el 6 de enero
  se corre al lunes por la Ley 51 de 1983 ni que el Corpus Christi depende de la
  Pascua. Un plazo mal contado cambia la respuesta jurídica.
- `buscar_normas` — el retrieval como herramienta.

**Criterio de invocación**: va en el prompt del sistema, no solo aquí. Si no
está en el prompt no es un criterio, es un deseo.

**Nunca adivina**: si no puede calcular, devuelve `calculable: False` con el
motivo, y el sistema lo dice.

In [13]:
from datetime import date

for a in (2024, 2025, 2026, 2027):
    f = terminos.festivos(a)
    colision = [(d.isoformat(), n) for d, n in f.items() if "/" in n]
    print(f"  {a}: {len(f)} festivos" + (f"   colision -> {colision}" if colision else ""))
print()
print("2025 tiene 17 y no 18 porque San Pedro (29/6, domingo -> 30/6) coincide con")
print("el Sagrado Corazon (Pascua + 71 = 30/6). La version anterior perdia uno en")
print("silencio al sobrescribir la clave del diccionario, y un festivo que")
print("desaparece corre TODOS los plazos del anio.")

  2024: 18 festivos
  2025: 17 festivos   colision -> [('2025-06-30', 'Sagrado Corazón / San Pedro y San Pablo')]
  2026: 18 festivos
  2027: 18 festivos

2025 tiene 17 y no 18 porque San Pedro (29/6, domingo -> 30/6) coincide con
el Sagrado Corazon (Pascua + 71 = 30/6). La version anterior perdia uno en
silencio al sobrescribir la clave del diccionario, y un festivo que
desaparece corre TODOS los plazos del anio.


In [14]:
print("La herramienta sobre casos reales, incluidos los que debe rechazar:\n")
for tipo, txt in [
    ("peticion_general",    "Radique una carta en la alcaldia hace dos meses"),
    ("peticion_documentos", "Pedi copia de mi historia clinica el 3 de marzo de 2026"),
    ("impugnacion",         "me notificaron el fallo el 12/10/2026"),
    ("peticion_general",    "Eso fue en marzo, estamos en junio"),
    ("fallo_tutela",        "no hay ninguna fecha aqui"),
    ("inventado",           "da igual"),
]:
    r = terminos.calcular_termino(tipo, relato=txt)
    if r["calculable"]:
        print(f"  OK      {tipo:<20} {r['fecha_inicio']} -> {r['fecha_vencimiento']}"
              f"  vencido={r['vencido']}")
    else:
        print(f"  RECHAZA {tipo:<20} {r['motivo'][:62]}")

La herramienta sobre casos reales, incluidos los que debe rechazar:

  OK      peticion_general     2026-08-22 -> 2026-09-11  vencido=True
  OK      peticion_documentos  2026-03-03 -> 2026-03-17  vencido=True
  OK      impugnacion          2026-10-12 -> 2026-10-15  vencido=True
  RECHAZA peticion_general     no se encontró una fecha legible en el relato
  RECHAZA fallo_tutela         no se encontró una fecha legible en el relato
  RECHAZA inventado            tipo de término desconocido: 'inventado'. Disponibles: ['cumpl


### Las cinco fallas del agente, provocadas a propósito

Documentar "esto nos pasó en una corrida" deja la cobertura al azar: si el
modelo no se equivoca ese día, la rama de manejo de la falla nunca se ejecuta y
nadie sabe si funciona. La autoprueba usa un cliente guionado, así que las cinco
se provocan de forma determinista y sin gastar un token.

In [15]:
print(subprocess.run([sys.executable, "src/autoprueba_agente.py"],
                     capture_output=True, text=True).stdout)

  AUTOPRUEBA DEL AGENTE · las cinco fallas, provocadas a proposito
escenario                           salida            mec.  pasos  contadores activados
------------------------------------------------------------------------------------------------
camino feliz                            ok          tutela      2  —
1 · argumentos no JSON                  ok          tutela      2  argumentos_no_json=1, paso_no_cobrado=1
2 · herramienta inexistente             ok          tutela      2  herramienta_inexistente=1, paso_no_cobrado=1
3 · herramienta que revienta            ok          tutela      2  herramienta_con_error=1, paso_no_cobrado=1
3b · tool externa con tipo invalido      ok          tutela      1  herramienta_con_error=1, casos_sin_ninguna_tool=1, paso_no_cobrado=1
4 · llamada repetida                    ok          tutela      2  llamada_repetida=1, paso_no_cobrado=1
5 · pasos agotados                      ok          tutela      2  llamada_repetida=3, pasos_agotados=1, bus

## 5 · El motor de reglas

El artículo 6 del Decreto 2591 no es un criterio difuso: son **cinco causales
tasadas que se aplican en orden**. El artículo 9 de la Ley 393 es una regla de
prioridad. El artículo 4 de la Ley 472 es una lista cerrada.

Pedirle a un modelo que *adivine* el resultado de aplicar reglas escritas es lo
que produce el colapso a "tutela". Aquí el LLM extrae **hechos** y el motor
decide: el enrutamiento es determinista, auditable, y la alucinación en el campo
`fundamento` cae a cero por construcción.

In [16]:
from reglas import Hechos, decidir, explicar

CASOS = {
 "E01": dict(afectacion_individual=True),
 "E31": dict(privacion_libertad=True),
 "A10": dict(privacion_libertad=True),
 "E09": dict(derecho_colectivo=True),
 "E28": dict(derecho_colectivo=True),
 "E29": dict(incumplimiento_normativo=True),
 "A09": dict(incumplimiento_normativo=True, afectacion_individual=True),
 "A11": dict(otro_medio_judicial=True, medio_eficaz=False, perjuicio_irremediable=True),
 "A01": dict(dano_consumado=True),
 "A07": dict(dano_consumado=True),
 "A12": dict(dano_consumado=True),
 "E10": dict(otro_medio_judicial=True, medio_eficaz=True),
 "A06": dict(otro_medio_judicial=True, medio_eficaz=True, sujeto_especial_proteccion=True),
 "M04": dict(otro_medio_judicial=True, medio_eficaz=True),
 "M06": dict(acto_general=True, afectacion_individual=False),
 "M08": dict(privacion_libertad=True),
 "M09": dict(incumplimiento_normativo=True, afectacion_individual=False),
 "M15": dict(derecho_colectivo=True, titular_pide_lo_suyo=True,
             perjuicio_irremediable=True, afectacion_individual=True),
}
todos = {e["id"]: e for e in schema.cargar() + eval_set + eval_m3}
ok, discrepan = 0, []
for cid, h in CASOS.items():
    d = decidir(Hechos(**h))
    bien = d.mecanismo == todos[cid]["mecanismo"]
    ok += bien
    if not bien:
        discrepan.append((cid, todos[cid]["mecanismo"], d.mecanismo))
print(f"el motor coincide con el gold en {ok}/{len(CASOS)} casos")
for cid, g, m in discrepan:
    print(f"  DISCREPA {cid}: gold={g} motor={m}")
print()
print("=== traza de M15: situacion colectiva pero el titular pide lo suyo ===")
print(explicar(decidir(Hechos(**CASOS["M15"]))))

el motor coincide con el gold en 18/18 casos

=== traza de M15: situacion colectiva pero el titular pide lo suyo ===
MECANISMO: tutela   (regla: tutela)
FUNDAMENTO: art. 86 CP

Reglas evaluadas, en orden:
  [no] art. 6.2 D.2591: no hay privación de la libertad
  [no] art. 6.3 D.2591: hay interés colectivo pero el titular pide lo suyo ante un perjuicio irremediable (excepción)
  [no] art. 6.5 D.2591: no se impugna un acto general sin aplicación concreta
  [no] art. 6.4 D.2591: no hay daño consumado, o la vulneración continúa (excepción del 6.4)
  [no] art. 9 L.393: no se reclama el cumplimiento de una norma
  [no] art. 6.1 D.2591: no existe otro medio de defensa judicial
  [SÍ] procedencia: ninguna causal cierra el paso


## 6 · Los cuatro brazos

In [17]:
generar_groq = sistema.generador_groq(
    __import__("groq").Groq(api_key=os.environ["GROQ_API_KEY"]),
    cache_path=c.ruta_cache("generacion.json"))

generar_qwen = None
if HAY_GPU:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    BASE = "Qwen/Qwen2.5-7B-Instruct"
    tok = AutoTokenizer.from_pretrained(BASE)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.bfloat16,
                                               device_map="cuda")
    mdl = PeftModel.from_pretrained(mdl, "models/m1-adaptador").eval()

    # Ver la nota de `recortar_al_bloque` en src/schema.py: el adaptador abre
    # turnos falsos del chat en una fraccion de los casos.
    IDS_PARADA_M3 = sorted({i for i in (
        tok.eos_token_id,
        tok.convert_tokens_to_ids("<|im_end|>"),
        tok.convert_tokens_to_ids("<|im_start|>"),
    ) if isinstance(i, int) and i >= 0 and i != tok.unk_token_id})
    RECORTES_M3: list[str] = []
    print("tokens de parada:", IDS_PARADA_M3)

    @torch.no_grad()
    def generar_qwen(sistema_txt, usuario):
        msgs = [{"role": "system", "content": sistema_txt},
                {"role": "user", "content": usuario}]
        p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = tok(p, return_tensors="pt", add_special_tokens=False).to(mdl.device)
        out = mdl.generate(**enc, max_new_tokens=220, do_sample=False,
                           eos_token_id=IDS_PARADA_M3,
                           pad_token_id=tok.pad_token_id)
        # TOKENS DEL MODELO LOCAL. `model.generate()` no devuelve `usage`,
        # y por eso los brazos 2 y 3 salieron con `llamadas_sin_usage: 90` y
        # `tokens_por_caso: null` en m3/20261019T173316Z: no era un fallo de
        # contabilidad, era que no habia nada que contar. El tokenizador si
        # los cuenta, exactos. Se publican en la propia funcion y cada brazo
        # los lee despues de SU llamada, asi que compartir el generador entre
        # los brazos 2 y 3 no mezcla su contabilidad.
        _n_entrada = int(enc["input_ids"].shape[1])
        generar_qwen.ultimos_tokens = (_n_entrada,
                                       int(out.shape[1]) - _n_entrada)
        bruto = tok.decode(out[0][enc["input_ids"].shape[1]:],
                           skip_special_tokens=True).strip()
        # Mismo recorte que en M1: el adaptador se descarrila en una fraccion
        # de los casos y la cola contamina el retrieval y las metricas RAGAS.
        texto, recortado = schema.recortar_al_bloque(bruto)
        if recortado:
            RECORTES_M3.append(usuario[:60])
        return texto

    print("modelo ajustado cargado desde models/m1-adaptador")
else:
    print("sin GPU: los brazos 2 y 3 se omiten")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

tokens de parada: [151644, 151645]
modelo ajustado cargado desde models/m1-adaptador


In [18]:
import time

CASOS_TODOS = eval_set + eval_m3
# ROADMAP #32. El fallo del brazo 4 casi se llevo 18 minutos de trabajo de los
# brazos 2 y 3, que estaban solo en memoria. Un brazo que ya corrio es un
# hecho: no tiene por que depender de que el siguiente no reviente.
#
# Se guarda tras CADA brazo y se relee al empezar, asi que volver a correr la
# celda retoma donde quedo en vez de repetir las llamadas que ya se pagaron.
import arranque

RUTA_CK = c.ruta_cache("brazos_checkpoint.json")
_ck = arranque.cargar_checkpoint(RUTA_CK)
brazos = _ck.get("brazos", {})
latencias = _ck.get("latencias", {})
fallos_por_brazo = _ck.get("fallos", {})
if brazos:
    print(f"checkpoint: {len(brazos)} brazo(s) ya calculado(s), se reutilizan")
    for _n in brazos:
        print(f"   {_n}")


def _ck_guardar():
    arranque.guardar_checkpoint(RUTA_CK, {"brazos": brazos,
                                          "latencias": latencias,
                                          "fallos": fallos_por_brazo})

preds_m1 = json.load(open("results/m1_predicciones.json", encoding="utf-8"))
# De QUE corrida de M1 salen estas predicciones. `publicar()` lo estampa en el
# archivo y desenvolver `datos` lo tiraba: la corrida de M3 documentaba datos,
# librerias, hardware y codigo, pero no cual de las corridas de M1 produjo las
# salidas del brazo 1 --- que es su linea base.
CORRIDA_M1 = preds_m1.get("_corrida") if isinstance(preds_m1, dict) else None
preds_m1 = preds_m1["datos"] if isinstance(preds_m1, dict) and "datos" in preds_m1 else preds_m1
assert [p["id"] for p in preds_m1] == [e["id"] for e in eval_set], "el orden no coincide"
print("predicciones de la corrida m1:", CORRIDA_M1 or "(sin registrar)")
# El brazo 1 solo existe para los casos del eval compartido: M1 nunca vio
# el eval de M3.
brazos["1 · fine-tuned solo (M1)"] = {e["id"]: p["finetuned"]
                                      for e, p in zip(eval_set, preds_m1)}

def _reloj(seg: float) -> str:
    m, s = divmod(int(seg), 60)
    return f"{m}m{s:02d}s" if m else f"{s}s"


def correr(sis, nombre, casos):
    """Corre un brazo sobre todos los casos, con ETA y sin morirse en el intento.

    DOS COSAS QUE LA PRIMERA VERSION NO HACIA

      - ETA. Los brazos tardan entre 6 y 10 minutos y la linea de progreso no
        decia cuanto faltaba, asi que no habia forma de distinguir "va lento"
        de "se colgo" sin esperar hasta el final.

      - Tolerar un fallo de API. Un 400 de Groq en UN caso tumbaba el brazo de
        61 entero, y con el los brazos anteriores que ya estaban en memoria.
        Paso de verdad con la extraccion de hechos del agente. El caso que
        falla queda con salida VACIA --- no con una respuesta por defecto, que
        entraria en las metricas como si el sistema hubiera contestado algo ---
        y se cuenta aparte para que el numero del brazo se lea sabiendo
        cuantos casos no se pudieron medir.
    """
    out, fallos, t0 = {}, [], time.perf_counter()
    n = len(casos)
    for i, e in enumerate(casos, 1):
        try:
            out[e["id"]] = sis(e["relato"])
        except Exception as ex:                       # noqa: BLE001
            fallos.append({"id": e["id"], "error": type(ex).__name__,
                           "detalle": str(ex)[:200]})
            out[e["id"]] = ""
        transcurrido = time.perf_counter() - t0
        eta = transcurrido / i * (n - i)
        print(f"  {nombre}: {i}/{n} · {transcurrido / i:.1f} s/caso · "
              f"transcurrido {_reloj(transcurrido)} · ETA {_reloj(eta)}   ",
              end="\r", flush=True)
    latencias[nombre] = (time.perf_counter() - t0) / n
    print(f"  {nombre}: {n}/{n} listo · {latencias[nombre]:.1f} s/caso · "
          f"total {_reloj(time.perf_counter() - t0)} · "
          f"{len(fallos)} fallos de API      ")
    for f in fallos:
        print(f"     FALLO {f['id']}: {f['error']} · {f['detalle']}")
    fallos_por_brazo[nombre] = fallos
    return out

s_ing = s_adv = None
# Valores por defecto ANTES del if: sin GPU los brazos 2 y 3 se omiten --- un
# camino contemplado a proposito --- pero `config_brazos`, que esta fuera del
# bloque, los lee igual. Sin esto, correr sin GPU moria con NameError al final,
# despues de haber hecho todo el trabajo del brazo 4.
elegida = decision["elegida"]
estrategia_ganadora, nombre_tr = None, "sin brazo 3 (no hay GPU)"
if generar_qwen is not None:
    s_ing = sistema.SistemaRAG(generar_qwen, indice, "ingenuo", k=5,
                               etiqueta="2 · fine-tuned + RAG ingenuo")
    # El brazo 3 tiene que desplegar LA TECNICA QUE GANO, no una parecida.
    # `SistemaRAG` hace `self.transformador(relato)`, asi que pasar `tr` a
    # secas invoca `tr.__call__`: la reescritura simple. Si la ganadora fue
    # multiconsulta o HyDE, se desplegaba una configuracion DISTINTA de la
    # medida --- y que ni siquiera esta en la tabla de recuperacion. Eso rompe
    # justo la cadena que M3 existe para demostrar: comparar, elegir, desplegar.
    estrategia_ganadora = elegida.split(" + ")[0]
    if "multiconsulta" in elegida:
        tr_brazo3, nombre_tr = (lambda r: tr.multiples(r, 3)), "multiconsulta (3)"
    elif "hyde" in elegida:
        tr_brazo3, nombre_tr = tr.hyde, "hyde"
    elif "reescritura" in elegida or "transformacion" in elegida:
        tr_brazo3, nombre_tr = tr, "reescritura simple"
    else:
        tr_brazo3, nombre_tr = None, "ninguna"
    print(f"brazo 3 despliega -> estrategia: {estrategia_ganadora} · "
          f"transformacion: {nombre_tr}   (ganadora: {elegida})")

    s_adv = sistema.SistemaRAG(generar_qwen, indice, estrategia_ganadora, k=5,
                               transformador=tr_brazo3,
                               etiqueta="3 · fine-tuned + RAG avanzado")
    if s_ing.etiqueta in brazos:
        print(f"  {s_ing.etiqueta}: ya estaba en el checkpoint, se salta")
    else:
        brazos[s_ing.etiqueta] = correr(s_ing, "brazo 2", CASOS_TODOS)
        _ck_guardar()
    if s_adv.etiqueta in brazos:
        print(f"  {s_adv.etiqueta}: ya estaba en el checkpoint, se salta")
    else:
        brazos[s_adv.etiqueta] = correr(s_adv, "brazo 3", CASOS_TODOS)
        _ck_guardar()

ag = agente.Agente(__import__("groq").Groq(api_key=os.environ["GROQ_API_KEY"]),
                   indice, estrategia="reranked", k=4, transformador=tr,
                   cache_path=c.ruta_cache("agente.json"),
                   etiqueta="4 · agente + reglas")
if ag.etiqueta in brazos:
    print(f"  {ag.etiqueta}: ya estaba en el checkpoint, se salta")
else:
    brazos[ag.etiqueta] = correr(ag, "brazo 4", CASOS_TODOS)
    _ck_guardar()
ag.guardar_cache()
tr.guardar()

ag.imprimir_resumen()
c.guardar_json("agente_resumen", ag.resumen())
c.guardar_json("config_brazos", {
    "corrida_m1_del_brazo_1": CORRIDA_M1,
    "ganadora_de_recuperacion": elegida,
    "estrategia_desplegada_brazo3": estrategia_ganadora,
    "transformacion_desplegada_brazo3": nombre_tr,
    "fallos_de_api_por_brazo": {k: len(v) for k, v in fallos_por_brazo.items()},
})
c.nota(f"brazo 1 desde la corrida m1 {CORRIDA_M1 or 'sin registrar'}")
c.nota(f"brazo 3 desplego {estrategia_ganadora} + {nombre_tr} "
       f"(ganadora: {elegida})")
c.guardar_json("agente_trazas", ag.trazas)

  checkpoint DESCARTADO: se genero con otro codigo o con otros datos.
     guardado: codigo f46a81cf6aab6793
     actual  : codigo 41ad18abecff8b82
  Reutilizarlo mezclaria brazos de dos codigos en una tabla.
predicciones de la corrida m1: 20261021T115654Z
brazo 3 despliega -> estrategia: hibrido · transformacion: multiconsulta (3)   (ganadora: hibrido + multiconsulta)
  brazo 2: 130/130 listo · 13.8 s/caso · total 29m56s · 0 fallos de API      
  brazo 3: 130/130 listo · 13.5 s/caso · total 29m11s · 0 fallos de API      
  brazo 4: 130/130 listo · 7.0 s/caso · total 15m13s · 0 fallos de API      

  TOOL USE · 4 · agente + reglas
  pasos/caso >= 4.15 (cota INFERIOR: 65 de 130 casos, 50%, se cortaron en el techo de 5 pasos)
  casos: 130 · pasos/caso: 4.15 · latencia 7.02s media, 12.29s p95

  contador                    veces
  ------------------------------------
  tool_invocada               614
  tool_buscar                 609
  tool_termino                5
  argumentos_no_json   

PosixPath('/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/agente_trazas.json')

## 7 · Criterio 4 — RAGAS y el harness, cruzados

Las cuatro métricas van implementadas en `src/ragas_eval.py`. La razón no es
evitar una dependencia: es que **dos de las cuatro se pueden calcular de forma
exacta**, porque el campo `fundamento` del gold dice qué artículos debían
recuperarse. Una medida exacta es mejor que una estimada, y se reportan las dos
para contrastarlas: si divergen mucho, el sospechoso es el juez.

La versión exacta tiene además una ventaja práctica: **no gasta tokens**, así
que cubre el 100% de los casos y no una ventana que se corta cuando se acaba el
cupo de la API.

In [19]:
from juez import Juez
j = Juez(cache=c.ruta_cache("juez.json"))

por_id = {ch["id"]: ch for ch in chunks}
contextos = {}
if s_ing:
    contextos[s_ing.etiqueta] = [t["recuperados"] for t in s_ing.trazas]
if s_adv:
    contextos[s_adv.etiqueta] = [t["recuperados"] for t in s_adv.trazas]
contextos[ag.etiqueta] = [t["recuperados"] for t in ag.trazas]

rs = []
for nombre, ids_por_caso in contextos.items():
    casos = CASOS_TODOS
    ctx = [[por_id[i] for i in ids if i in por_id] for ids in ids_por_caso]
    sal = [brazos[nombre][e["id"]] for e in casos]
    rs.append(ragas_eval.evaluar(nombre, sal, ctx, casos,
                                 relatos=[e["relato"] for e in casos], juez=j))
ragas_eval.imprimir(rs)

# B6 · LAS DOS COLUMNAS NO SE PUEDEN LEER POR SEPARADO, Y HAY QUE DECIRLO
# DENTRO DEL ARTEFACTO, NO SOLO EN LA CONSOLA.
#
# Medido en m3/20261016T215955Z: `faithfulness` exacta daba 0.589 para el
# brazo 2 y el juez 0.143 --- un factor de 4 --- mientras en
# `context_precision` el juez era MAS generoso (0.460 contra 0.393). No es un
# desfase de calibracion que se pueda corregir con un offset: es divergencia,
# y reportar cualquiera de las dos sola induciria una conclusion distinta.
#
# Y el nombre del artefacto invita a creer que lo calculo la libreria `ragas`.
# Si no esta instalada, son implementacion propia, y eso cambia como se leen.
import importlib.util as _ilu
_lib = _ilu.find_spec("ragas") is not None
for _r in rs:
    _e, _jz = _r.get("exactas", {}), _r.get("juez", {})
    _r["_implementacion"] = ("libreria ragas" if _lib
                             else "implementacion propia (src/ragas_eval.py); "
                                  "la libreria `ragas` NO esta instalada")
    _r["_divergencia_exacta_vs_juez"] = {
        k: round(_jz[k] - _e[k], 4)
        for k in set(_e) & set(_jz)}
    _r["_lectura"] = (
        "`exactas` se calcula contra el gold con reglas deterministas; `juez` "
        "lo puntua un LLM. Divergen, y no con un desfase constante: en esta "
        "corrida el juez fue mucho mas severo en faithfulness y mas generoso "
        "en context_precision. Ninguna de las dos se puede reportar sola. "
        "Coherente con la dimension 2 de M2, donde el juez coincide con otro "
        "juez (kappa 0.76) mucho mejor que con la verdad de dominio (0.26).")

print()
print(f"  implementacion: {rs[0]['_implementacion']}")
print("  divergencia juez - exacta, por brazo:")
for _r in rs:
    _d = _r["_divergencia_exacta_vs_juez"]
    print(f"    {_r['sistema']:<32}"
          + "  ".join(f"{k}={v:+.3f}" for k, v in sorted(_d.items())))
print("  Signos opuestos entre metricas = divergencia, no calibracion.")

c.guardar_json("ragas", rs)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


  RAGAS · exacto (contra el gold) vs estimado (juez)
sistema                     faithf.  ctx prec  ctx rec  | faithf.  ctx prec  ans rel
------------------------------------------------------------------------------
2 · fine-tuned + RAG ingenuo   32.8%     36.0%    31.9%     11.3%     41.9%    39.1%
3 · fine-tuned + RAG avanzado   55.1%     61.7%    51.6%     21.1%     71.5%    38.1%
4 · agente + reglas          35.0%     50.8%    56.3%     18.2%     52.1%    38.9%
  exacto = contra el `fundamento` del gold · estimado = lo que juzga el LLM
  Si divergen mucho, el sospechoso es el juez, no el sistema.

  implementacion: libreria ragas
  divergencia juez - exacta, por brazo:
    2 · fine-tuned + RAG ingenuo    context_precision=+0.059  faithfulness=-0.215
    3 · fine-tuned + RAG avanzado   context_precision=+0.099  faithfulness=-0.340
    4 · agente + reglas             context_precision=+0.013  faithfulness=-0.168
  Signos opuestos entre metricas = divergencia, no calibracion.


PosixPath('/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/ragas.json')

### La lectura cruzada: ¿los aciertos son del retrieval?

La pregunta incómoda: un caso puede acertar **sin que el contexto recuperado
respalde la respuesta**. Si eso pasa, parte del acierto no se puede atribuir al
retrieval y el sistema parece mejor de lo que es.

Se mide caso por caso con `context_recall` exacto, que no cuesta tokens.

In [20]:
nombre_ag = ag.etiqueta
ctx_ag = [[por_id[i] for i in ids if i in por_id] for ids in contextos[nombre_ag]]

filas_cruce = []
for e, ctx in zip(CASOS_TODOS, ctx_ag):
    sal = brazos[nombre_ag][e["id"]]
    p = schema.parse_salida(sal)
    acierta = (p.get("mecanismo") == e["mecanismo"])
    filas_cruce.append({
        # `evaluable` es False en los casos de abstencion, que NO tienen
        # mecanismo gold. Sin esta columna, `acierta=False` en esas filas se
        # lee como "fallo" cuando significa "no aplica", y el CSV publicado no
        # da forma de distinguirlo.
        "id": e["id"], "tipo": e["tipo"],
        "evaluable": e["mecanismo"] is not None, "acierta": acierta,
        "context_recall": round(ragas_eval.context_recall_exacto([ctx], [e]), 3),
        "context_precision": round(ragas_eval.context_precision_exacta([ctx], [e]), 3),
        "faithfulness": round(ragas_eval.faithfulness_exacta([sal], [ctx]), 3),
    })

# EL DELTA VA SOBRE LOS EVALUABLES, Y ESTE ES EL CASO QUE LO PRUEBA.
#
# Los casos de abstencion no tienen mecanismo gold, asi que `acierta` es False
# en todos ellos por construccion --- y sus metricas RAGAS son 0.0, porque no
# hay citas esperadas contra las que medir. Meterlos en el grupo "fallan"
# arrastra ese grupo hacia cero y infla el delta.
#
# Medido sobre m3/20261016T215955Z, con las 10 filas de abstencion dentro:
#
#     context_recall      +0.168   contra  +0.037  real     (4.5x)
#     context_precision   +0.213   contra  +0.104  real     (2x)
#     faithfulness        +0.012   contra  -0.018  real     (CAMBIA DE SIGNO)
#
# El tercero es el que obliga a arreglarlo: la tabla afirmaba "mas fiel cuando
# acierta" y el dato dice lo contrario. Se imprimen las dos columnas para que
# la diferencia quede a la vista en vez de quedar en un comentario.
evaluables = [f for f in filas_cruce if f["evaluable"]]
abstencion = [f for f in filas_cruce if not f["evaluable"]]
ac = [f for f in evaluables if f["acierta"]]
fa = [f for f in evaluables if not f["acierta"]]
_todo_ac = [f for f in filas_cruce if f["acierta"]]
_todo_fa = [f for f in filas_cruce if not f["acierta"]]


def _med(rows, m):
    return sum(f[m] for f in rows) / len(rows) if rows else 0.0


print(f"  evaluables {len(evaluables)} · abstencion {len(abstencion)} "
      f"(sin mecanismo gold, fuera del delta)")
print()
print(f"  {'metrica':<20}{'aciertan':>10}{'fallan':>10}{'delta':>10}"
      f"{'si entrara abstencion':>24}")
print("  " + "-" * 74)
for m in ("context_recall", "context_precision", "faithfulness"):
    a, b = _med(ac, m), _med(fa, m)
    sesgado = _med(_todo_ac, m) - _med(_todo_fa, m)
    marca = "  <-- cambia de signo" if (a - b) * sesgado < 0 else ""
    print(f"  {m:<20}{a:>10.3f}{b:>10.3f}{a - b:>+10.3f}"
          f"{sesgado:>+24.3f}{marca}")
print(f"  {'':<20}{'n=' + str(len(ac)):>10}{'n=' + str(len(fa)):>10}")
print()
print("  La ultima columna NO es un resultado: es lo que daria metiendo las")
print("  filas de abstencion, y esta ahi para que no se vuelva a colar.")
print()
sin_respaldo = [f["id"] for f in ac if f["context_recall"] == 0]
print(f"ACIERTOS CON context_recall = 0: {len(sin_respaldo)} de {len(ac)}")
print(f"  {sin_respaldo}")
print()
if sin_respaldo:
    print("Esos aciertos NO se pueden atribuir al retrieval: el contexto no")
    print("contenia lo que respalda la respuesta. Si solo se midiera con el")
    print("harness, el sistema pareceria mejor de lo que es.")
    print("En el brazo 4 la explicacion es el motor de reglas: el mecanismo lo")
    print("deriva el art. 6 del Decreto 2591, no el texto recuperado. Es un")
    print("acierto legitimo pero de otra fuente, y conviene decir cual.")
else:
    print("Todos los aciertos tienen respaldo en el contexto recuperado.")
c.guardar_csv("cruce_ragas_harness", filas_cruce)

  evaluables 120 · abstencion 10 (sin mecanismo gold, fuera del delta)

  metrica               aciertan    fallan     delta   si entrara abstencion
  --------------------------------------------------------------------------
  context_recall           0.585     0.581    +0.004                  +0.047
  context_precision        0.530     0.454    +0.076                  +0.105
  faithfulness             0.314     0.380    -0.066                  -0.082
                            n=86      n=34

  La ultima columna NO es un resultado: es lo que daria metiendo las
  filas de abstencion, y esta ahi para que no se vuelva a colar.

ACIERTOS CON context_recall = 0: 14 de 86
  ['E08', 'E19', 'E21', 'E29', 'E34', 'A07', 'A12', 'A22', 'A23', 'A36', 'M04', 'M10', 'M14', 'M15']

Esos aciertos NO se pueden atribuir al retrieval: el contexto no
contenia lo que respalda la respuesta. Si solo se midiera con el
harness, el sistema pareceria mejor de lo que es.
En el brazo 4 la explicacion es el motor

PosixPath('/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/cruce_ragas_harness.csv')

### Contraste con la librería `ragas` real

Las métricas implementadas a mano se contrastan con la librería sobre una
muestra. Si divergen, el sospechoso somos nosotros.

In [21]:
# El LLM con el que ragas juzga. Se declara aqui y no se deja al azar de la
# omision de la libreria: una metrica cuyo juez no se sabe cual fue no se
# puede comparar con nada.
from juez import MODELO as _MODELO_JUEZ
MODELO_JUEZ_RAGAS = _MODELO_JUEZ

# La instalacion esta en la CELDA 1, con su porque. Aqui solo se usa: un
# `pip install` a estas alturas actualizaria paquetes que `torch`, `datasets`
# y `sentence_transformers` ya importaron, y el fallo saldria como un error de
# atributo dentro de ragas --- lejos de su causa y despues de 40 minutos de
# brazos.

try:
    from ragas import evaluate as ragas_evaluate
    from ragas.metrics import faithfulness as m_faith
    from datasets import Dataset as HFDataset
    import ragas as ragas_lib

    MUESTRA = 8
    casos_m = CASOS_TODOS[:MUESTRA]
    ctx_m = ctx_ag[:MUESTRA]
    filas = {
        "question": [e["relato"] for e in casos_m],
        "answer": [brazos[nombre_ag][e["id"]] for e in casos_m],
        "contexts": [[f"[{x['articulo']} {x['norma']}] {x['texto']}" for x in cx]
                     for cx in ctx_m],
        "ground_truth": [schema.render_salida(e) for e in casos_m],
    }
    # EL LLM VA EXPLICITO. `ragas.evaluate()` sin `llm=` apunta a OpenAI por
    # omision, y este proyecto solo tiene clave de Groq: con el pin correcto
    # habria fallado en autenticacion en vez de en el import, que es el mismo
    # resultado --- la celda sin correr --- por otra causa. Groq sirve un
    # endpoint compatible con OpenAI, asi que se le pasa apuntando ahi.
    from langchain_openai import ChatOpenAI
    from ragas.llms import LangchainLLMWrapper

    _llm = LangchainLLMWrapper(ChatOpenAI(
        model=MODELO_JUEZ_RAGAS,
        base_url="https://api.groq.com/openai/v1",
        api_key=os.environ["GROQ_API_KEY"],
        temperature=0))
    print(f"ragas {ragas_lib.__version__} sobre {MUESTRA} casos, "
          f"juzgando con {MODELO_JUEZ_RAGAS} via Groq...")
    res = ragas_evaluate(HFDataset.from_dict(filas), metrics=[m_faith],
                         llm=_llm)
    print("librería :", res)
    nuestro = ragas_eval.faithfulness_exacta(filas["answer"], ctx_m)
    print(f"nuestro faithfulness EXACTO sobre los mismos {MUESTRA}: {nuestro:.3f}")
    print()
    print("No son la misma cantidad y no deberian coincidir: la libreria le")
    print("pregunta a un LLM si el contexto sostiene cada afirmacion, y el")
    print("nuestro comprueba si cada CITA esta respaldada por el contexto. El")
    print("contraste sirve para ver si apuntan en la misma direccion.")
    c.guardar_json("ragas_libreria", {
        "muestra": MUESTRA, "resultado": str(res), "nuestro_exacto": nuestro,
        "version_ragas": ragas_lib.__version__,
        "llm_juez_de_ragas": MODELO_JUEZ_RAGAS,
        "lectura": (
            "La libreria y la metrica propia NO miden lo mismo y no deberian "
            "coincidir: la libreria le pregunta a un LLM si el contexto "
            "sostiene cada afirmacion; la nuestra comprueba si cada CITA esta "
            "respaldada por el contexto. El contraste sirve para ver si "
            "apuntan en la misma direccion, no para validar una con la otra.")})
except Exception as ex:
    print("no se pudo correr la libreria:", str(ex)[:300])
    print("Las metricas exactas y con juez de la celda anterior NO dependen de esto.")
    c.guardar_json("ragas_libreria", {
        "error": str(ex)[:300], "pin_intentado": RAGAS_PIN,
        "llm_juez_intentado": MODELO_JUEZ_RAGAS,
        "lectura": (
            "El contraste contra la libreria no se pudo correr. Las metricas "
            "RAGAS que el reporte publica son implementacion propia "
            "(`src/ragas_eval.py`) y no dependen de esto; lo que falta es el "
            "contraste que diria si apuntan en la misma direccion.")})

no se pudo correr la libreria: No module named 'langchain_community.chat_models.vertexai'
Las metricas exactas y con juez de la celda anterior NO dependen de esto.


## 8 · El harness de M2, sin un solo cambio

In [22]:
scorecards = []
for nombre, salidas_por_id in brazos.items():
    casos = eval_set if nombre.startswith("1 ") else CASOS_TODOS
    sal = [salidas_por_id[e["id"]] for e in casos]
    sc = harness.tabla_puente(casos, sal, nombre, juez=j)
    sc["latencia_s_por_caso"] = round(latencias.get(nombre, 0.0), 2)
    scorecards.append(sc)
    metricas.imprimir(sc)

  juez 109/109 · 908 llamadas, 2470 desde caché, 0 ilegibles

  SCORECARD · 1 · fine-tuned solo (M1)  (n=103)
campo           accuracy  macro-F1  mayorit.     delta
--------------------------------------------------------------
derecho           63.1%     61.5%     14.6%    +48.5%
mecanismo         71.8%     79.2%     44.7%    +27.2%
procede           81.6%     81.2%     55.3%    +26.2%
--------------------------------------------------------------
adherencia estructural (5 etiquetas) 100.0%
adherencia de vocabulario             89.3%
coherencia procede<->mecanismo       100.0%
ROUGE-L (prosa)                       30.0%
similitud embeddings                  83.9%
acierto jurídico (mecanismo+procede)  71.8%
se abstuvo donde debía               100.0%   (6/6)
ABSTENCIÓN INDEBIDA                    8.7%   (9/103 casos CON respuesta)
   se callo en A01 (indemnizatorio, gold via_ordinaria)
   se callo en A04 (contra_providencia, gold via_ordinaria)
   se callo en E34 (contractual_ordinario

### 8.1 · Los cuatro brazos sobre LOS MISMOS casos

La tabla anterior pone en columnas contiguas cosas que no son comparables: el
brazo 1 se puntúa **solo** sobre el eval compartido —M1 nunca vio el eval de
M3— y los brazos 2 a 4 sobre el compartido **más** el de M3. Compararlos así
castiga a los brazos con RAG por haber sido evaluados además en un conjunto
más duro, y encima cada columna trae una clase mayoritaria distinta, así que
ni el listón es el mismo.

Esta celda vuelve a puntuar los cuatro **sobre los casos comunes**. Es la
tabla que hay que citar para decir cuál es mejor; la anterior sirve para ver
cómo se comporta cada brazo en todo lo que puede responder.

Los dos tamaños no se escriben aquí a propósito: cambian con cada versión del
eval set, y una cifra clavada en el markdown caduca sin que nadie se entere
—ya pasó en esta misma celda—. La salida de la celda los imprime, y la celda
de publicación los repite al lado de cada CSV diciendo cuál es cuál.

In [23]:
COMUNES = [e["id"] for e in eval_set]
# Conteos derivados, nunca cableados: el eval set crecio de 46 a 69 casos
# y los numeros escritos a mano en los mensajes quedaron mintiendo.
N_GOLD = sum(e["tipo"] == "gold" for e in eval_set)
N_JUZGABLES = sum(e["mecanismo"] is not None for e in eval_set)
scorecards_comunes = []
for nombre, salidas_por_id in brazos.items():
    if any(i not in salidas_por_id for i in COMUNES):
        print(f"  {nombre}: no cubre los {len(COMUNES)} comunes, se omite")
        continue
    sc = metricas.scorecard([salidas_por_id[i] for i in COMUNES],
                            eval_set, nombre)
    scorecards_comunes.append(sc)

print(f"{'brazo':<34}{'n':>4}{'acc_mec':>9}{'macroF1':>9}{'juridico':>10}"
      f"{'gold':>8}{'advers':>8}")
print("-" * 82)
for sc in scorecards_comunes:
    pt = sc.get("por_tipo", {})
    print(f"{sc['sistema']:<34}{sc['n']:>4}"
          f"{sc['campos']['mecanismo']['accuracy']:>9.3f}"
          f"{sc['campos']['mecanismo']['macro_f1']:>9.3f}"
          f"{sc['acierto_juridico']:>10.3f}"
          f"{pt.get('gold', {}).get('acierto_juridico', 0):>8.3f}"
          f"{pt.get('adversarial', {}).get('acierto_juridico', 0):>8.3f}")

# El ganador NO es el mismo segun el tipo de caso, y esa es la conclusion.
mejor_glob = max(scorecards_comunes, key=lambda s: s["acierto_juridico"])
mejor_gold = max(scorecards_comunes,
                 key=lambda s: s["por_tipo"].get("gold", {}).get("acierto_juridico", 0))
mejor_adv = max(scorecards_comunes,
                key=lambda s: s["por_tipo"].get("adversarial", {}).get("acierto_juridico", 0))
print()
print(f"  mejor en agregado   : {mejor_glob['sistema']}")
print(f"  mejor en gold       : {mejor_gold['sistema']}")
print(f"  mejor en adversarial: {mejor_adv['sistema']}")
if mejor_gold["sistema"] != mejor_adv["sistema"]:
    print()
    print("  No hay un ganador unico: el agregado esta dominado por los")
    print(f"  casos gold ({N_GOLD} de {N_JUZGABLES} juzgables), asi que premia")
    print("  al brazo que ya los resolvia. En los adversariales --- las")
    print("  trampas, donde el fine-tuning solo es debil --- gana otro.")
    print("  Reportar solo el agregado esconde eso.")
c.guardar_json("scorecards_comunes", scorecards_comunes)
c.guardar_json("ganadores", {
    "subconjunto": f"los {len(COMUNES)} casos de {ARCHIVO_EVAL}, "
                   f"comunes a los cuatro brazos",
    "agregado": mejor_glob["sistema"],
    "gold": mejor_gold["sistema"],
    "adversarial": mejor_adv["sistema"]})

brazo                                n  acc_mec  macroF1  juridico    gold  advers
----------------------------------------------------------------------------------
1 · fine-tuned solo (M1)           103    0.718    0.792     0.718   0.781   0.615
2 · fine-tuned + RAG ingenuo       103    0.602    0.577     0.592   0.781   0.282
3 · fine-tuned + RAG avanzado      103    0.631    0.605     0.602   0.688   0.462
4 · agente + reglas                103    0.728    0.686     0.728   0.688   0.795

  mejor en agregado   : 4 · agente + reglas
  mejor en gold       : 1 · fine-tuned solo (M1)
  mejor en adversarial: 4 · agente + reglas

  No hay un ganador unico: el agregado esta dominado por los
  casos gold (64 de 103 juzgables), asi que premia
  al brazo que ya los resolvia. En los adversariales --- las
  trampas, donde el fine-tuning solo es debil --- gana otro.
  Reportar solo el agregado esconde eso.


PosixPath('/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/ganadores.json')

In [24]:
metricas.comparar(scorecards[0], scorecards[-1])

print()
print(f"{'sistema':<34}{'acc_mec':>9}{'gold':>8}{'advers':>8}{'abstenc':>9}{'s/caso':>8}")
print("-" * 76)
for sc in scorecards:
    ab = sc.get("abstencion")
    # `acierto_abstencion` es None cuando el conjunto no trae casos sin
    # respuesta correcta, asi que no basta comprobar que `ab` exista.
    ab_txt = ("n/a" if not ab or ab.get("acierto_abstencion") is None
              else f"{ab['acierto_abstencion']:.0%}")
    print(f"{sc['sistema']:<34}{sc['campos']['mecanismo']['accuracy']:>9.1%}"
          f"{sc['por_tipo'].get('gold',{}).get('acierto_juridico',0):>8.1%}"
          f"{sc['por_tipo'].get('adversarial',{}).get('acierto_juridico',0):>8.1%}"
          f"{ab_txt:>9}{sc['latencia_s_por_caso']:>8.1f}")


  1 · fine-tuned solo (M1)  ->  4 · agente + reglas
métrica                      antes   después     delta
--------------------------------------------------------------
derecho (acc)               63.1%     65.8%+    +2.7%
mecanismo (acc)             71.8%     71.7%-    -0.2%
procede (acc)               81.6%     79.2%-    -2.4%
adherencia estructural     100.0%    100.0%     +0.0%
adherencia vocabulario      89.3%     97.5%+    +8.2%
ROUGE-L (prosa)             30.0%     22.1%-    -8.0%
acierto jurídico            71.8%     71.7%-    -0.2%
alucinación norm.            0.0%      0.0%     +0.0%
coherencia                 100.0%    100.0%     +0.0%
similitud embeddings        83.9%     80.1%-    -3.9%
precisión de citas          55.1%     44.2%-   -10.9%
alucinación (salida)         0.0%      0.0%     +0.0%
EMPEORÓ en: mecanismo (acc), procede (acc), ROUGE-L (prosa), acierto jurídico, similitud embeddings, precisión de citas  <- reportar esto, no esconderlo

sistema                    

### 8.2 · Qué cuesta cada brazo

La tabla de calidad compara aciertos. Esta compara **lo que cuesta
conseguirlos**, y sin ella no se puede decidir nada: el brazo 4 tarda ~8 s por
caso contra ~0 del brazo 1, y eso no es un detalle de implementación — es la
conclusión.

Tres cosas que la media de latencia escondía:

- la **p50** frente a la media: el agente hace entre 1 y 5 pasos según el caso,
  así que su distribución tiene cola y la media la arrastran los casos que
  agotan el presupuesto;
- las **llamadas reales** frente a las de caché: una segunda corrida sale
  entera de caché y parece gratis, cuando el coste real es el de la primera;
- los **tokens**, que es lo que se factura y lo que topa el cupo de la API.

El brazo 1 aparece como `n/d`, no como 0: son predicciones de M1, su coste es
el de esa corrida, y un 0 se leería como «es gratis».

In [25]:
import costos

_sistemas = [s for s in (s_ing, s_adv, ag) if s is not None]


class _BrazoUno:
    """El brazo 1 no tiene coste medible AQUI: son predicciones de M1."""
    etiqueta = "1 · fine-tuned solo (M1)"


filas_costos = costos.tabla([_BrazoUno()] + _sistemas)
costos.imprimir(filas_costos)
c.guardar_json("costos", filas_costos)


  COSTE POR BRAZO
  brazo                          casos  llam.  l/caso   p50 s   p95 s  tok/caso
  ----------------------------------------------------------------------------------
  1 · fine-tuned solo (M1)         n/d    n/d     n/d     n/d     n/d       n/d
  2 · fine-tuned + RAG ingenuo     130    130     1.0  13.665  16.095    2190.8
  3 · fine-tuned + RAG avanzado    130    130     1.0  13.002  16.521    2149.4
  4 · agente + reglas              130    369    2.84   1.664   4.018   13090.6
  ----------------------------------------------------------------------------------
  478 llamadas salieron de cache: no cuentan ni latencia ni tokens.
  n/d = no se midio aqui. El brazo 1 son predicciones de M1: su coste
  es el de esa corrida, no cero.


PosixPath('/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/costos.json')

In [26]:
# Los dos CSV se escriben primero: la guarda de abajo los comprueba igual que
# a los demas, y `scorecard_comunes` no existe hasta que `a_csv` lo escribe.
harness.a_csv(scorecards, str(c.dir / "scorecard.csv"))
c.artefactos["scorecard"] = "scorecard.csv"
c.guardar_json("scorecards", scorecards)
c.guardar_json("salidas", brazos)

# `m3_scorecard.csv` pone el brazo 1 con n=63 al lado de los brazos 2-4 con
# n=80, y con base mayoritaria distinta (0.4286 contra 0.45): son conjuntos
# distintos, porque M1 no tiene predicciones sobre `eval_set_m3.json`. La
# tabla sobre los COMUNES es la que se puede leer como comparacion, y vivia
# solo en la carpeta de la corrida.
harness.a_csv(scorecards_comunes, str(c.dir / "scorecard_comunes.csv"))
c.artefactos["scorecard_comunes"] = "scorecard_comunes.csv"

# ======================================================================
# GUARDA DE PUBLICACION · comprobar TODOS antes de publicar NINGUNO
# ======================================================================
# El numero no se escribe aqui: sale de len(A_PUBLICAR), para que anadir
# un artefacto no deje este comentario mintiendo.
#
# `publicar()` falla en el primero que no existe, y falla a mitad. Medido en
# m3/20261017T024911Z: dejo 6 artefactos publicados y 6 sin publicar, con
# `results/` en un estado que no corresponde a ninguna corrida completa ---
# justo lo que `verificar_entrega.py` existe para detectar, pero provocado
# por la propia celda de publicacion.
#
# Y el mensaje decia "no existe el artefacto 'cruce_ragas_harness'" sin decir
# QUE CELDA lo genera. Con 44 celdas y la publicacion a diez de distancia,
# eso obliga a buscar. Aqui se nombra.
A_PUBLICAR = [
    ("scorecard", "results/m3_scorecard.csv"),
    ("costos", "results/m3_costos.json"),
    ("retrieval", "results/m3_retrieval.json"),
    ("ragas", "results/m3_ragas.json"),
    ("chunking", "results/m3_chunking.json"),
    ("agente_resumen", "results/m3_agente.json"),
    ("cruce_ragas_harness", "results/m3_cruce.csv"),
    ("scorecard_comunes", "results/m3_scorecard_comunes.csv"),
    ("scorecards_comunes", "results/m3_scorecards_comunes.json"),
    ("ganadores", "results/m3_ganadores.json"),
    ("decision_retrieval", "results/m3_decision_retrieval.json"),
    ("ragas_libreria", "results/m3_ragas_libreria.json"),
    # Las trazas del agente: es la evidencia que ROADMAP3 M3-1 pide leer
    # para saber que hechos extrae en los 8 casos de
    # `accion_cumplimiento`, y se quedaba en la carpeta de la corrida.
    ("agente_trazas", "results/m3_agente_trazas.json"),
]
QUIEN_LO_GENERA = {
    "chunking": "seccion 2 · la ablacion de chunking",
    "retrieval": "seccion 3 · la ablacion de recuperacion",
    "decision_retrieval": "seccion 3 · la eleccion de estrategia",
    "agente_resumen": "seccion 6 · el brazo 4, el del agente",
    "ragas": "seccion 7 · las metricas RAGAS",
    "ragas_libreria": "seccion 7 · las metricas RAGAS",
    "cruce_ragas_harness": "seccion 7 · el CRUCE (la que construye `filas_cruce`)",
    "scorecards_comunes": "seccion 8 · la tabla sobre los casos COMUNES",
    "ganadores": "seccion 8 · la tabla sobre los casos COMUNES",
    "costos": "seccion 8 · la tabla de costos",
    "agente_trazas": "seccion 6 - el brazo 4, el del agente",
}
print(f"  comprobando {len(A_PUBLICAR)} artefactos antes de publicar")
faltan = [n for n, _ in A_PUBLICAR
          if not (c.dir / c.artefactos.get(n, f"{n}.json")).exists()]
if faltan:
    print("=" * 70)
    print(f"  NO SE PUBLICA NADA. Faltan {len(faltan)} de {len(A_PUBLICAR)} "
          f"artefactos:")
    print("=" * 70)
    for n in faltan:
        print(f"    {n:<22} lo genera -> "
              f"{QUIEN_LO_GENERA.get(n, 'revisa `c.artefactos`')}")
    print()
    print("  Corre esas celdas y vuelve a correr esta. No se publica nada a")
    print("  medias: `results/` con 6 de 12 no corresponde a ninguna corrida.")
    print("=" * 70)
    raise RuntimeError(f"faltan {len(faltan)} artefactos: {', '.join(faltan)}")

print(f"  los {len(A_PUBLICAR)} artefactos estan. Publicando.")
for _n, _destino in A_PUBLICAR:
    c.publicar(_n, _destino)

print()
print("  CUAL ES CUAL, porque se publican las dos:")
print(f"    m3_scorecard_comunes.csv   n={scorecards_comunes[0]['n']} en los "
      f"cuatro brazos  <- COMPARABLE")
print(f"    m3_scorecard.csv           n mixta ("
      f"{', '.join(str(s['n']) for s in scorecards)})  <- cada brazo sobre su "
      f"propio conjunto")
print("    Para afirmar que un brazo es mejor que otro, la de los comunes.")

mejor_sc = max(scorecards, key=lambda s: s["acierto_juridico"])
c.cerrar(valida=bool(ag.contadores["tool_invocada"]), notas=[
    f"chunking elegido: {dec_chunk['elegido']}",
    f"retrieval elegido: {decision['elegida']} "
    f"(descalificadas: {decision['descalificadas'] or 'ninguna'})",
    # Sobre los COMUNES y desglosado por tipo: el agregado solo esta
    # dominado por los gold y afirmar con el un "mejor brazo" unico es
    # justamente lo que esconde el hallazgo.
    f"sobre los {len(COMUNES)} comunes -> agregado: {mejor_glob['sistema']} · "
    f"gold: {mejor_gold['sistema']} · adversarial: {mejor_adv['sistema']}",
    f"(sobre su propio conjunto, mejor agregado: {mejor_sc['sistema']} "
    f"con acierto juridico {mejor_sc['acierto_juridico']:.4f})",
    f"herramientas invocadas: {ag.contadores['tool_invocada']}, "
    f"fallas manejadas: {sum(v for k, v in ag.contadores.items() if k.startswith(('argumentos','herramienta','llamada','pasos')))}",
    f"aciertos sin respaldo en el contexto: {len(sin_respaldo)}",
])

import pandas as pd
pd.read_csv("results/m3_scorecard.csv").set_index("sistema").T

/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/scorecard.csv · 4 filas · 31 columnas
/content/drive/MyDrive/Colab Notebooks/agenteAmparo/runs/m3/20261021T140413Z/scorecard_comunes.csv · 4 filas · 24 columnas
  comprobando 13 artefactos antes de publicar
  los 13 artefactos estan. Publicando.
  publicado -> results/m3_scorecard.csv  (de m3/20261021T140413Z)
  publicado -> results/m3_costos.json  (de m3/20261021T140413Z)
  publicado -> results/m3_retrieval.json  (de m3/20261021T140413Z)
  publicado -> results/m3_ragas.json  (de m3/20261021T140413Z)
  publicado -> results/m3_chunking.json  (de m3/20261021T140413Z)
  publicado -> results/m3_agente.json  (de m3/20261021T140413Z)
  publicado -> results/m3_cruce.csv  (de m3/20261021T140413Z)
  publicado -> results/m3_scorecard_comunes.csv  (de m3/20261021T140413Z)
  publicado -> results/m3_scorecards_comunes.json  (de m3/20261021T140413Z)
  publicado -> results/m3_ganadores.json  (de m3/20261021T140413Z)
  public

sistema,1 · fine-tuned solo (M1),2 · fine-tuned + RAG ingenuo,3 · fine-tuned + RAG avanzado,4 · agente + reglas
n,103.0000,120.0000,120.0000,120.0000
acc_derecho,0.6311,0.2750,0.2833,0.6583
acc_mecanismo,0.7184,0.6250,0.6583,0.7167
acc_procede,0.8155,0.6917,0.6917,0.7917
macroF1_mecanismo,0.7919,0.6104,0.6452,0.6458
baseline_mayoritaria,0.4466,0.4583,0.4583,0.4583
rouge_l,0.3002,0.2688,0.2628,0.2206
similitud_embeddings,0.8394,0.8117,0.8006,0.8007
adherencia_estructural,1.0000,1.0000,1.0000,1.0000
adherencia_vocabulario,0.8932,0.8833,0.8250,0.9750


## 9 · Dónde falla el sistema

Rellenar con los números de la corrida. Lo que hay que mirar y no maquillar:

1. **El delta de recuperación, con la guarda de seguridad.** ¿Qué técnica movió
   qué? Si alguna quedó descalificada por recuperar menos que el ingenuo, es un
   resultado, no un fracaso: HyDE y multi-consulta cuestan una llamada a la API
   por caso y hay que saber si la pagan.

2. **La latencia.** Está en la tabla de deltas y en la última columna del
   scorecard. Una técnica que gana dos puntos de recall y multiplica por 180 el
   tiempo de retrieval no entra al sistema.

3. **`acc_mecanismo` contra la clase mayoritaria.** Es el listón real. El brazo
   4 debería despegarse de forma clara, porque el mecanismo no lo predice: lo
   calcula una regla.

4. **Los adversariales.** La medida de si el sistema razona o reconoce formas.
   El salto del brazo 1 al 4 es el resultado central de M3.

5. **La abstención, en los dos sentidos.** `acierto_abstencion` sola no dice
   nada: un sistema que se calla siempre saca 100%. Hay que leerla junto a
   `abstencion_indebida`.

6. **Las dos métricas de cita.** `tasa_alucinacion` dice si la norma existe;
   `precision_citas` dice si viene al caso. Un sistema puede tener 0% de
   alucinación y citas inservibles.

7. **Los aciertos con `context_recall = 0`.** Son aciertos que no vienen del
   retrieval. En el brazo 4 la explicación es el motor de reglas, y hay que
   decirlo: el mérito es del art. 6 del Decreto 2591, no de la búsqueda.

8. **La abstención del brazo 4, que ahora existe.** Antes daba 0 de 6 por
   construcción: el motor siempre encontraba una vía
   (`sistema.DEFECTO_PRO_HOMINE`). Ahora el extractor responde además si hay
   un caso que triar, y si no lo hay el brazo emite la abstención del formato.
   El número que hay que mirar **no** es el acierto sobre los 6 —seis casos no
   validan ningún criterio de abstención— sino `abstencion_indebida` sobre los
   103 **con** respuesta correcta: valía 0.0, y si el criterio nuevo se
   abstiene de más, sube. Y el contador `abstencion_por_no_triable` dice
   cuántas veces se usó, para que la cifra no se lea sin saber de dónde sale.

9. **El presupuesto del agente.** `pasos_promedio` está **censurado por
   arriba**: un caso que se corta en el techo no aporta los pasos que habría
   necesitado, aporta el techo, así que la media estima una cota *inferior*.
   Con más de la mitad de la muestra tocando el techo no estima nada útil, y
   `casos_en_el_techo` es la cifra que hay que leer primero. El presupuesto
   cuenta ahora pasos **productivos** —una búsqueda cuyos fragmentos ya
   estaban todos en las observaciones anteriores no cobra— y hay un techo de
   iteraciones aparte, porque «dar vueltas sin aportar» y «necesitar más
   pasos» son cosas distintas y antes se contaban igual.

### Limitaciones, sin adornos

- **Corpus:** 69 fragmentos · 20 de la Constitución · 17 del Decreto 2591.
  Cubre los casos de evaluación y no la casuística real de la tutela
  colombiana, y está sesgado hacia procedimiento. (Esa línea la comprueba
  `data/auditar_notebooks.py` contra `data/corpus.json`: la versión anterior
  decía 57 fragmentos y 19 de la Constitución, y llevaba dos ampliaciones del
  corpus sin actualizarse.)
- **Eval:** 109 casos en el compartido y 21 en el de M3. Con ese tamaño una
  diferencia de 5 puntos está dentro del ruido. Los deltas grandes son
  fiables; los pequeños, no, y el intervalo de confianza de cada scorecard es
  lo que lo dice —no el orden de la tabla—.
- **Las etiquetas las puso un equipo sin formación jurídica.** 142 de las 160
  del dataset las deriva el motor de reglas —todas menos las 18 de abstención,
  que por diseño no llevan hechos—, lo que las hace **auditables, no
  correctas**. Ya apareció un error así (A09).
- **El brazo 4 depende de que la extracción de hechos sea correcta.** Si el LLM
  se equivoca al decir si existe otro medio judicial, la regla aplica
  impecablemente sobre un hecho falso. La decisión es auditable, que no es lo
  mismo que correcta.
- **Esto no es apto para uso real.** Es un ejercicio académico. Un sistema que
  decide si alguien accede a la justicia necesita validación de abogados.

## 10 · Cierre de la entrega · los cuatro pasos de después

M3 es el último de los tres, así que aquí se cierra todo. Los cuatro pasos van
en una sola celda **a propósito**: por separado es fácil correr uno y olvidar
los otros, y el que se olvida siempre es el que habría avisado de algo.

Ninguno de los cuatro escribe en `results/`, así que esta celda no puede
estropear la corrida que acaba de terminar. Lo peor que puede pasar es que
diga que algo no cuadra — que es su trabajo.

| paso | qué hace | si falla |
|---|---|---|
| `verificar_entrega` | que los 33 artefactos estén, sean de la misma corrida de su módulo y compartan huella de código | **lo importante**: las cifras no son del mismo experimento y no se pueden comparar |
| `reporte` | genera `REPORTE.md` leyendo los artefactos, sin escribir ninguna cifra a mano | se niega a escribir si falta un artefacto, en vez de dejar huecos silenciosos |
| `diagnostico_cumplimiento` | lee `m3_agente_trazas.json` y dice **cuál** de las 6 causas rompe `accion_cumplimiento` | queda acotado a 6 en vez de resuelto |
| `calibracion_juez --preparar` | escribe `data/calibracion_1.csv` con 10 casos para que una persona los califique | falta M1 o M2, y lo dice nombrando el archivo |


In [27]:
# Los cuatro pasos de cierre, en una sola celda.
#
# POR QUE JUNTOS Y NO SUELTOS
#
# Son los pasos de "ya acabo, ¿quedo bien?", y ese es justo el momento en que
# se dan por hechos. `verificar_entrega.py` existia desde el ROADMAP v1 y no se
# habia corrido NI UNA VEZ: lo que comprueba se venia haciendo a mano, leyendo
# los `corrida.json`. Una comprobacion que hay que acordarse de hacer es una
# comprobacion que no se hace.
#
# NINGUNO escribe en `results/`, asi que esta celda no puede estropear la
# corrida. Y ninguno corta la celda si falla: se corren los cuatro y se resume
# al final, porque enterarse de los cuatro problemas de una vez es mejor que
# enterarse del primero cuatro veces.
import subprocess
import sys as _sys

PASOS = [
    (["data/verificar_entrega.py"],
     "artefactos, corridas y huellas",
     "la entrega NO es coherente: o faltan artefactos, o no son "
     "del mismo experimento. Las cifras no se pueden comparar"),
    (["data/reporte.py"],
     "genera REPORTE.md desde los artefactos",
     "el reporte no se escribio: falta algun artefacto"),
    (["data/diagnostico_cumplimiento.py"],
     "cual de las 6 causas rompe accion_cumplimiento",
     "no se pudo leer las trazas del agente"),
    (["data/calibracion_juez.py", "--preparar"],
     "formulario de calibracion (10 casos, para una persona)",
     "falta M1 o M2"),
]

_res = []
for _args, _que, _si_falla in PASOS:
    print()
    print("=" * 74)
    print(f"  {_args[0].split('/')[-1]} · {_que}")
    print("=" * 74)
    _r = subprocess.run([_sys.executable, "-I"] + _args,
                        capture_output=True, text=True)
    print((_r.stdout or "").rstrip()[-4000:])
    if _r.stderr.strip():
        print("  stderr:", _r.stderr.strip()[-600:])
    _res.append((_args[0], _r.returncode, _si_falla))

print()
print("#" * 74)
print("  RESUMEN DEL CIERRE")
print("#" * 74)
for _a, _c, _si in _res:
    _n = _a.split("/")[-1]
    print(f"  {'OK   ' if _c == 0 else 'FALLA'} {_n:32s}" +
          ("" if _c == 0 else f"  <- {_si}"))
_malos = [a for a, c, _ in _res if c != 0]
print()
if not _malos:
    print("  Los cuatro pasan. La entrega es coherente, REPORTE.md esta escrito")
    print("  y data/calibracion_1.csv espera a que una persona lo califique.")
else:
    print(f"  {len(_malos)} paso(s) con problemas. Arriba esta el detalle de")
    print("  cada uno, con el archivo o la cifra que lo causa.")
print("#" * 74)
print()
print("  LO QUE SIGUE, Y NO SE PUEDE HACER AQUI:")
print("  · Bajar data/calibracion_1.csv y dárselo a la MISMA persona externa")
print("    que hizo la doble anotacion. Son 10 casos, ~15 min.")
print("  · Al devolverlo:  python -I data/calibracion_juez.py --medir \\")
print("                      --anotador persona-externa")
print("    Con `persona-externa` la cifra es validacion externa del juez; con")
print("    `equipo` o `asistente` el artefacto escribe que NO lo es.")


  verificar_entrega.py · artefactos, corridas y huellas
  VERIFICACION DE LA ENTREGA

  1 · artefactos esperados por modulo
    M1   12/12
    M2   11/11
    M3   13/13

  2 · coherencia de corrida dentro de cada modulo
    M1   12 artefacto(s), 1 corrida: 20261021T115654Z
    M2   11 artefacto(s), 1 corrida: 20261021T124137Z
    M3   13 artefacto(s), 1 corrida: 20261021T140413Z

  3 · huella de codigo y version de datos entre modulos
    codigo: la misma huella en 3 modulo(s) (41ad18abecff8b82)
    datos: eval_set 1.2.0, n=46 en 3 modulo(s)

  4 · copias sueltas en runs/
    1 archivo(s) en la RAIZ de runs/, fuera de cualquier carpeta de corrida:
      cache_juez.json

  AVISO  1 copia(s) suelta(s) en runs/. Asi aparecieron las 9 con los numeros del dataset v2: estan en un sitio plausible y no son las vigentes.

  La entrega esta completa y es coherente: los artefactos existen,
  vienen de la misma corrida de su modulo, y los tres modulos
  usaron el mismo codigo y la misma version d